# 飞书 Token 全自动获取

运行下面这个 cell,自动完成：启动服务器 → 打开浏览器 → 扫码授权 → 收回调 → 换 t. ken → 保存到本地。

In [ ]:
import os, json, time, socket, secrets, threading, webbrowser, requests
from pathlib import Path
from urllib.parse import urlencode, parse_qs, urlparse
from http.server import HTTPServer, BaseHTTPRequestHandler

# ========== 配置 ==========
APP_ID = os.environ.get("FEISHU_APP_ID", "cli_xxxxxxxxxxxxxxxx")
APP_SECRET = os.environ.get("FEISHU_APP_SECRET", "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
PORT = 8088
TIMEOUT = 120
SCOPE = "offline_access"
BASE_URL = "https://open.feishu.cn/open-apis"
REDIRECT_URI = f"http://localhost:{PORT}"

if "xxxxxxxx" in APP_ID:
    raise ValueError("请先设置 APP_ID 和 APP_SECRET(直接改代码或设环境变量)")

# ========== 1. 生成授权链接 ==========
state = secrets.token_urlsafe(16)
auth_url = (
    f"https://accounts.feishu.cn/open-apis/authen/v1/authorize?"
    f"app_id={APP_ID}"
    f"&redirect_uri={requests.utils.quote(REDIRECT_URI)}"
    f"&scope={requests.utils.quote(SCOPE)}"
    f"&state={state}"
)

# ========== 2. 回调处理器(收到 code 后自动换 token)==========
result = {"done": False, "data": None, "error": None}

class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        pass

    def do_GET(self):
        if self.path == "/favicon.ico":
            self.send_response(404); self.end_headers(); return

        q = parse_qs(urlparse(self.path).query)
        code, returned_state, error = q.get("code", [None])[0], q.get("state", [None])[0], q.get("error", [None])[0]

        if error:
            result["error"] = f"授权被拒绝: {error}"; result["done"] = True
            self._html(f"<h2 style='color:red'>❌ 授权被拒绝: {error}</h2>", 400); return

        if not code or returned_state != state:
            result["error"] = "缺少 code 或 state 不匹配"; result["done"] = True
            self._html("<h2 style='color:red'>❌ 参数错误</h2>", 400); return

        # 用 code 换 token
        try:
            resp = requests.post(
                f"{BASE_URL}/authen/v2/oauth/token",
                headers={"Content-Type": "application/json; charset=utf-8"},
                json={"grant_type": "authorization_code", "client_id": APP_ID,
                      "client_secret": APP_SECRET, "code": code, "redirect_uri": REDIRECT_URI},
                timeout=30)
            data = resp.json()
            if data.get("code", -1) != 0:
                err = data.get("error_description", "Unknown")
                result["error"] = f"换 token 失败: {err}"; result["done"] = True
                self._html(f"<h2 style='color:red'>❌ 换 token 失败</h2><pre>{err}</pre>", 400); return

            # 保存 token
            result["data"] = data
            acc, ref, exp = data["access_token"], data.get("refresh_token", "无"), data["expires_in"]
            cache_path = Path.home() / f".feishu_oauth_{APP_ID[-8:]}.json"
            cache = {"app_id": APP_ID, "access_token": acc, "refresh_token": ref,
                     "expire_at": time.time() + exp, "scope": data.get("scope", ""), "saved_at": time.time()}
            cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")

            html = f"""<!DOCTYPE html>
            <html><head><meta charset="utf-8"><title>授权成功</title>
            <style>body{{font-family:sans-serif;max-width:700px;margin:40px auto;padding:20px}}
            .ok{{color:green}} .box{{background:#f5f5f5;padding:15px;border-radius:8px;margin:15px 0}}
            code{{background:#e0e0e0;padding:2px 6px;border-radius:4px;word-break:break-all}}
            </style></head><body>
            <h2 class="ok">✅ 授权成功！Token 已获取并保存</h2>
            <div class="box">
                <p><b>access_token:</b> <code>{acc[:50]}...</code> ({len(acc)} 字符)</p>
                <p><b>refresh_token:</b> <code>{ref[:50] if ref != "无" else "未获取"}...</code></p>
                <p><b>有效期:</b> {exp}s (约 {exp//3600} 小时)</p>
                <p><b>保存路径:</b> <code>{cache_path}</code></p>
            </div>
            <p>可以关闭此页面,回到 N. tebook 查看结果。</p>
            <script>
                console.log("[Feishu] access_token:", "{acc}");
                console.log("[Feishu] refresh_token:", "{ref if ref != "无" else "N/A"}");
                console.log("[Feishu] expires_in:", {exp});
            </script></body></html>"""
            result["done"] = True
            self._html(html)
        except Exception as e:
            result["error"] = str(e); result["done"] = True
            self._html(f"<h2 style='color:red'>❌ 异常</h2><pre>{e}</pre>", 500)

    def _html(self, body, status=200):
        self.send_response(status)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.end_headers()
        self.wfile.write(body.encode("utf-8"))

# ========== 3. 先启动服务器 ==========
class ReusableServer(HTTPServer):
    allow_reuse_address = True

server = ReusableServer(("127.0.0.1", PORT), Handler)
t = threading.Thread(target=server.serve_forever, daemon=True)
t.start()

time.sleep(0.3)
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(2)
try:
    sock.connect(("127.0.0.1", PORT))
    sock.close()
    print(f"✅ 服务器已就绪: http://127.0.0.1:{PORT}")
except Exception:
    print("⚠️ 服务器启动中...")

# ========== 4. 再打开浏览器 ==========
print(f"\n📎 授权链接:\n   {auth_url}\n")
try:
    webbrowser.open(auth_url)
    print("🌐 已打开浏览器,请扫码授权...\n")
except Exception:
    print("⚠️ 请手动复制上方链接到浏览器\n")

# ========== 5. 等待回调 ==========
start = time.time()
while not result["done"] and (time.time() - start) < TIMEOUT:
    time.sleep(0.5)
    if int(time.time() - start) % 5 == 0 and int(time.time() - start) > 0 and not result["done"]:
        print(f"   已等待 {int(time.time() - start)}s...")

try:
    server.shutdown(); server.server_close()
except Exception:
    pass

# ========== 6. 结果 ==========
if result["error"]:
    print(f"\n❌ {result['error']}")
elif not result["done"]:
    pri. t("\n❌ 超时。请检查是否已完成授权,以及安全设置中. 否添加了重定向 URL。")
else:
    d = result["data"]
    print("\n✅ 完成！Token 已保存到本地缓存")
    print(f"   access_token:  {d['access_token'][:30]}...")
    print(f"   refresh_token: {d.get('refresh_token', '无')[:30] if d.get('refresh_token') else '无'}...")
    print(f"   有效期: {d['expires_in']}s")
    print(f"   缓存文件: {Path.home() / f'.feishu_oauth_{APP_ID[-8:]}.json'}")
    print(f"\n💡 浏览器页面 F12 → Console 可查看完整 token")


---

## 手动刷新 Token(无需浏览器)

如果已有 `refresh_token`,运行下面 cell 自动续期：

In [1]:
import json, time, requests
from pathlib import Path

APP_ID = os.environ.get("FEISHU_APP_ID", "cli_xxxxxxxxxxxxxxxx")
APP_SECRET = os.environ.get("FEISHU_APP_SECRET", "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
BASE_URL = "https://open.feishu.cn/open-apis"
cache_path = Path.home() / f".feishu_oauth_{APP_ID[-8:]}.json"

if not cache_path.exists():
    raise FileNotFoundError(f"没有找到缓存文件: . cache_path}。请先. 行上面的授权 cell。")

cache = json.loads(cache_path.read_text(encoding="utf-8"))
refresh_token = cache.get("refresh_token")
if not refresh_token:
    raise ValueError("缓存中没有 refresh_. oken,无法自动续期。")

print("🔄 正在刷新 token...")
resp = requests.post(
    f"{BASE_URL}/authen/v2/oauth/token",
    headers={"Content-Type": "application/json; charset=utf-8"},
    json={"grant_type": "refresh_token", "client_id": APP_ID,
          "client_secret": APP_SECRET, "refresh_token": refresh_token},
    timeout=30)
data = resp.json()

if data.get("code", -1) != 0:
    print(f"❌ 刷新失败: {data.get('error_description')} (code: {data.get('code')})")
else:
    acc, ref, exp = data["access_token"], data.get("refresh_token"), data["expires_in"]
    cache["access_token"] = acc
    cache["refresh_token"] = ref
    cache["expire_at"] = time.time() + exp
    cache["saved_at"] = time.time()
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
    print("✅ 刷新成功并已保存！")
    print(f"   access_token:  {acc[:30]}...")
    print(f"   refresh_token: {ref[:30] if ref else '无'}...")
    print(f"   有效期: {exp}s")


d:\python-envs\baseenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


NameError: name 'os' is not defined